In [1]:
!pip install clip

  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-0.2.0-py3-none-any.whl size=6989 sha256=fc1355d3554673126b64d0d1b104a9a4c78842f840cc01250bbc937fcb348761
  Stored in directory: /root/.cache/pip/wheels/6c/fd/54/9d4e15cf829b871199a7cd3597e869a514d1624a0a43076896
Successfully built clip


# Helper Functions

In [2]:
# setup libraries and functions

import torch
if torch.cuda.is_available():
    print("GPU is available.")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}") # Assuming at least one GPU
else:
    print("GPU is not available.")

import sys
!{sys.executable} -m pip install av

from diffusers import AutoencoderKLCogVideoX
from diffusers.utils import export_to_video
import gc
import numpy as np

import torchvision
from torchvision import transforms

torch.cuda.empty_cache()
gc.collect()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load 3d vae
# use float16 to save memory
model_id = "THUDM/CogVideoX-2b"
vae = AutoencoderKLCogVideoX.from_pretrained(
    model_id,
    subfolder='vae',
    torch_dtype=torch.float16,
).to(device)

# enables tiling which should save on vram usage
vae.enable_tiling()

# helper to prep video for vae
def load_and_process_video(video_path, height=480, width=720, max_frames=16):
    print(f"Loading {video_path}...")

    # read video
    video_frames, _, _ = torchvision.io.read_video(video_path, output_format="TCHW", pts_unit='sec')

    # cap frames
    if len(video_frames) > max_frames:
        video_frames = video_frames[:max_frames]

    current_frames = len(video_frames)

    # transform pipeline
    transform = transforms.Compose([
        transforms.Resize(height, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop((height, width)),
    ])

    # apply transforms
    processed_frames = torch.stack([transform(f) for f in video_frames])

    # format for VAE
    video_tensor = processed_frames.permute(1, 0, 2, 3).unsqueeze(0)

    # normalize
    video_tensor = video_tensor.float() / 255.0  # Now [0, 1]
    video_tensor = (video_tensor * 2.0) - 1.0    # Now [-1, 1]

    return video_tensor.to(device, dtype=torch.float16)

# manually splits vid into temporal chunks to save vram
def get_latents_chunked(video_tensor, chunk_size=4):
    frames = video_tensor.shape[2]
    latent_list = []

    with torch.no_grad():
        for i in range(0, frames, chunk_size):
            # get slice of frame
            end = min(i + chunk_size, frames)
            video_chunk = video_tensor[:,:,i:end,:,:]

            # encode current chunk
            posterior = vae.encode(video_chunk).latent_dist
            latents = posterior.sample() * vae.config.scaling_factor
            latent_list.append(latents)

            # clean up vram
            del video_chunk, posterior, latents
            torch.cuda.empty_cache()

    # stitch chunks together and return
    return torch.cat(latent_list, dim=2)


# decode latents & manually split to save vram
def decode_latents_chunked(latents, chunk_size=1):
    latent_frames_count = latents.shape[2]
    decoded_video_list = []

    with torch.no_grad():
        for i in range(0, latent_frames_count, chunk_size):
            # slice latent
            end = min(i+chunk_size, latent_frames_count)
            latent_chunk = latents[:,:,i:end,:,:]

            # decode current chunk
            frames = vae.decode(latent_chunk).sample

            # move to cpu to free vram
            frames = (frames / 2 + 0.5).clamp(0,1)
            decoded_video_list.append(frames.cpu())

            # cleanup
            del latent_chunk, frames
            torch.cuda.empty_cache()
    # stitch together and return
    return torch.cat(decoded_video_list, dim=2)


GPU is available.
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA L4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 65.7 MB/s eta 0:00:00


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

# Load Dataset of Averages

In [3]:
# Load the .pt file
class_average_latents = torch.load('/content/class_average_latents.pt')
classes = list(class_average_latents.keys())

print(f"Loaded {len(classes)} classes")
print(classes)

Loaded 13 classes
['yoga', 'texting', 'welding', 'bartending', 'zumba', 'laughing', 'cartwheeling', 'motorcycling', 'archery', 'sailing', 'dodgeball', 'jogging', 'headbanging']


# Generate Videos of the Latent Space Averages (Visualized)

In [4]:
from IPython.display import HTML
from base64 import b64encode
OUTPUT_FILE = 'output.mp4'

# change class to visualize here
class_name = classes[9]

vid_latent = class_average_latents[class_name]

decoded_frames = decode_latents_chunked(vid_latent)

video_tensor = decoded_frames[0]
video_tensor = video_tensor.permute(1,2,3,0)
video_np = video_tensor.cpu().numpy()
video_np = (video_np * 255).astype(np.uint8)

export_to_video(video_np, 'output.mp4', fps=16)

mp4 = open(OUTPUT_FILE,'rb').read()
print(f'Latent Average of {class_name}')
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

Latent Average of sailing


# Now, show differences between two averages.

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
from diffusers.utils import export_to_video

class_A = classes[3]
class_B = classes[9]
num_frames = 64
fps = 16
output_file = "latent_morph.mp4"

latent_A = class_average_latents[class_A].to(device)
latent_B = class_average_latents[class_B].to(device)

if latent_A.dim() == 4:
    latent_A = latent_A.unsqueeze(0)
    latent_B = latent_B.unsqueeze(0)

B, C, T0, H, W = latent_A.shape
print("Latent shape:", latent_A.shape)

# Interpolation
half = torch.linspace(0, 1, num_frames // 2, device=device)
alphas = torch.cat([half, half.flip(0)])

# per frame latents
frames = []
for a in alphas:
    blended = (1 - a) * latent_A + a * latent_B
    frames.append(blended)

# concatonate along time
vid_latent = torch.cat(frames, dim=2)

print("Interpolated latent:", vid_latent.shape)

# temporal downsampling
vid_latent = F.interpolate(
    vid_latent,
    size=(num_frames, H, W),
    mode="trilinear",
    align_corners=False
)

print("Resampled latent:", vid_latent.shape)

# temporal smoothing
def temporal_smooth(latent, kernel_size=7):
    pad = kernel_size // 2

    kernel = torch.ones(
        kernel_size,
        device=latent.device,
        dtype=latent.dtype
    ) / kernel_size

    kernel = kernel.view(1, 1, -1, 1, 1)

    latent = F.pad(latent, (0,0,0,0,pad,pad), mode="replicate")

    return F.conv3d(
        latent,
        kernel.repeat(latent.shape[1], 1, 1, 1, 1),
        groups=latent.shape[1]
    )


vid_latent = temporal_smooth(vid_latent)
print("Smoothed latent:", vid_latent.shape)
decoded = decode_latents_chunked(vid_latent, chunk_size=1)

video_tensor = decoded[0]
video_tensor = video_tensor.permute(1,2,3,0)
video_np = (video_tensor.cpu().numpy() * 255).astype(np.uint8)

export_to_video(video_np, output_file, fps=fps)

print(f"Saved video: {output_file}")
print(f"Classes: {class_A} ↔ {class_B}")


Latent shape: torch.Size([1, 16, 4, 60, 90])
Interpolated latent: torch.Size([1, 16, 256, 60, 90])
Resampled latent: torch.Size([1, 16, 64, 60, 90])
Smoothed latent: torch.Size([1, 16, 64, 60, 90])
Saved video: latent_morph.mp4
Classes: bartending ↔ sailing


In [6]:
from IPython.display import HTML
from base64 import b64encode

OUTPUT_FILE = "latent_morph.mp4"

mp4 = open(OUTPUT_FILE, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width="480" controls loop>
  <source src="{data_url}" type="video/mp4">
</video>
""")

This represents the differences between latent averages between concepts 'bartending' and 'sailing'. This demonstrates things like average color and movement directions. This gradually (like a slider or a scale) shows how as you are closer/at one concept, it has a different average representation as the other.

# Videos depicting representations of all classes.
This is basicaally just noisy averages of colors and stuff for each class.

In [7]:
import torch
import torch.nn as nn
import numpy as np
import gc

i = 0

while(i < 12):
  class_name = classes[i]
  i =+ 1
  num_frames = 64
  pca_components = 8
  gru_hidden = pca_components
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # load class latent, ensure shape (1, C, T, H, W)
  orig = class_average_latents[class_name]
  if orig.dim() == 4:
      orig = orig.unsqueeze(0)
  B, C, T0, H, W = orig.shape
  print("Orig latent shape:", orig.shape)

  # build sample matrix for PCA: flatten (C*H*W) per time step, optionally augment by small jitter
  with torch.no_grad():
      orig_cpu = orig.cpu().float().squeeze(0)
      frames_list = []
      for t in range(T0):
          v = orig_cpu[:, t, :, :].reshape(-1)
          frames_list.append(v)
      X = torch.stack(frames_list, dim=0)

      n_aug = max(0, 64 - X.shape[0])
      if n_aug > 0:
          reps = (n_aug + X.shape[0] - 1) // X.shape[0]
          X_rep = X.repeat(reps, 1)[:n_aug]
          noise = 1e-3 * torch.randn_like(X_rep)
          X = torch.cat([X, X_rep + noise], dim=0)

      mean_vec = X.mean(dim=0, keepdim=True)
      Xc = X - mean_vec

      # SVD to get principal directions (works when n_samples <= dim)
      U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
      comps = Vh[:pca_components].contiguous()
      comps = comps.float()
      mean_vec = mean_vec.squeeze(0).float()

  # Build tiny GRUCell on CPU (very small, no GPU VRAM usage)
  gru = nn.GRUCell(input_size=pca_components, hidden_size=gru_hidden).cpu()
  # init hidden by projecting the first frame (or mean of original frames)
  first_frame = orig_cpu[:, 0, :, :].reshape(-1).float()
  proj0 = (first_frame - mean_vec) @ comps.T
  hidden = proj0.unsqueeze(0)

  # small random input sequence (zeros ok)
  seq = []
  inp = torch.zeros(1, pca_components, dtype=torch.float32)
  for t in range(num_frames):
      hidden = gru(inp, hidden)
      seq.append(hidden.squeeze(0).clone())
  seq = torch.stack(seq, dim=0)

  # reconstruct flattened latents: recon = mean + coords @ comps
  recon_flat = mean_vec.unsqueeze(0) + (seq @ comps)

  # reshape to (1, C, num_frames, H, W)
  dim = C * H * W
  assert recon_flat.shape[1] == dim, "Dimension mismatch during reshape"
  recon = recon_flat.view(num_frames, C, H, W)
  recon = recon.permute(1, 0, 2, 3).unsqueeze(0)

  # convert to device and dtype expected by vae (float16 on GPU) and decode chunked
  recon = recon.to(device)
  if device == 'cuda':
      recon = recon.to(torch.float16)
  else:
      recon = recon.to(torch.float32)

  del orig_cpu, X, Xc, U, S, Vh, comps, mean_vec, seq, recon_flat
  gc.collect()
  torch.cuda.empty_cache()

  decoded = decode_latents_chunked(recon, chunk_size=1)
  video_tensor = decoded[0].permute(1,2,3,0)
  video_np = (video_tensor.cpu().numpy() * 255).astype(np.uint8)

  from diffusers.utils import export_to_video
  output_file = f"pca_gru_{class_name.replace(' ','_')}.mp4"
  export_to_video(video_np, output_file, fps=16)
  print("Saved:", output_file)

  from IPython.display import HTML
  from base64 import b64encode
  mp4 = open(output_file,'rb').read()
  data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
  HTML(f"""<video width=480 controls loop><source src="{data_url}" type="video/mp4"></video>""")
  x =+ 1

Orig latent shape: torch.Size([1, 16, 4, 60, 90])
Saved: pca_gru_yoga.mp4
Orig latent shape: torch.Size([1, 16, 4, 60, 90])
Saved: pca_gru_texting.mp4
Orig latent shape: torch.Size([1, 16, 4, 60, 90])
Saved: pca_gru_texting.mp4
Orig latent shape: torch.Size([1, 16, 4, 60, 90])
Saved: pca_gru_texting.mp4
Orig latent shape: torch.Size([1, 16, 4, 60, 90])


KeyboardInterrupt: 

# Now lets look at the addition of two videos and see what we can get from this.

In [9]:
video_a = load_and_process_video('/content/king_walk.mp4', max_frames=64)
video_b = load_and_process_video('/content/queen_walk.mp4', max_frames=64)

Loading /content/king_walk.mp4...
Loading /content/queen_walk.mp4...


In [14]:
latents_a = get_latents_chunked(video_a)
latents_b = get_latents_chunked(video_b)

In [15]:
latents_mixture = latents_a + latents_b
latents_difference = latents_a - latents_b

In [31]:
print('decoding hybrid vid')
decoded_frames = decode_latents_chunked(latents_mixture)

video_tensor = decoded_frames[0]
video_tensor = video_tensor.permute(1,2,3,0)
video_np = video_tensor.cpu().numpy()
video_np = (video_np * 255).astype(np.uint8)

print('saving vid')
export_to_video(video_np, 'output_addition.mp4', fps=16)

print('decoding hybrid vid2')
decoded_frames2 = decode_latents_chunked(latents_difference)

video_tensor2 = decoded_frames2[0]
video_tensor2 = video_tensor2.permute(1,2,3,0)
video_np2 = video_tensor2.cpu().numpy()
video_np2 = (video_np2 * 255).astype(np.uint8)

print('saving vid2')
export_to_video(video_np2, 'output_differences.mp4', fps=16)

decoding hybrid vid
saving vid
decoding hybrid vid2
saving vid2


'output_differences.mp4'

In [32]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('output_addition.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [33]:

from IPython.display import HTML
from base64 import b64encode
mp4 = open('output_differences.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

# Pipeline:

In [54]:
import torch
import numpy as np
from base64 import b64encode
from IPython.display import HTML, display

EPS = 1e-6
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def compute_stats(z):
    mean = z.mean(dim=(2,3,4), keepdim=True)
    std  = z.std(dim=(2,3,4), keepdim=True)
    return mean, std

def rescale_to_target(z, target_mean, target_std, eps=EPS):
    m, s = compute_stats(z)
    return (z - m) / (s + eps) * (target_std + eps) + target_mean

def anchor_color(lat_edit, lat_anchor):
    mean_a, std_a = compute_stats(lat_anchor)
    return rescale_to_target(lat_edit, mean_a, std_a)

def interpolate_latents(lat_a, lat_b, alpha=0.5):
    lat_mix = (1 - alpha) * lat_a + alpha * lat_b
    mean_a, std_a = compute_stats(lat_a)
    mean_b, std_b = compute_stats(lat_b)
    target_mean = (1 - alpha) * mean_a + alpha * mean_b
    target_std  = (1 - alpha) * std_a  + alpha * std_b
    return rescale_to_target(lat_mix, target_mean, target_std)

def suppress_high_freq_channels(z, keep_ratio=0.7):
    C = z.shape[1]
    keep = int(C * keep_ratio)
    out = z.clone()
    out[:, keep:] *= 0.0
    return out

def temporal_smooth(z):
    return 0.25 * z[:, :, :-2] + 0.5 * z[:, :, 1:-1] + 0.25 * z[:, :, 2:]

def decode_and_save(latents, filename, decode_func, export_func, fps=16):
    latents = latents.to(DEVICE).contiguous()
    with torch.no_grad():
        decoded = decode_func(latents)

    dec0 = decoded[0]
    video = dec0.permute(1,2,3,0).clamp(0,1).cpu().numpy()
    video = (video * 255).astype(np.uint8)
    export_func(video, filename, fps=fps)
    return filename

def run_pipeline(video_king_path, video_queen_path, direction_amount,
                 max_frames=64,
                 fps=16,
                 decode_func=None,
                 export_func=None
                 ):

    video_k = load_and_process_video(video_king_path, max_frames=max_frames)
    video_q = load_and_process_video(video_queen_path, max_frames=max_frames)

    lat_k = get_latents_chunked(video_k).to(DEVICE)
    lat_q = get_latents_chunked(video_q).to(DEVICE)

    assert lat_k.shape == lat_q.shape

    direction = lat_q - lat_k

    lat_king = lat_k - (direction_amount * direction)
    lat_king = anchor_color(lat_king, lat_k)
    lat_king = suppress_high_freq_channels(lat_king)

    lat_mid = interpolate_latents(lat_k, lat_q, alpha=0.5)
    lat_mid = anchor_color(lat_mid, lat_k)
    lat_mid = suppress_high_freq_channels(lat_mid)

    lat_queen = lat_q + (direction_amount * direction)
    lat_queen = anchor_color(lat_queen, lat_q)
    lat_queen = suppress_high_freq_channels(lat_queen)

    out_k = decode_and_save(lat_king,  'output_king.mp4',  decode_func, export_func, fps)
    out_m = decode_and_save(lat_mid,   'output_mid.mp4',   decode_func, export_func, fps)
    out_q = decode_and_save(lat_queen, 'output_queen.mp4', decode_func, export_func, fps)

    return out_k, out_m, out_q


In [55]:
out_king, out_mid, out_queen = run_pipeline(
    video_king_path="/content/king_walk.mp4",
    video_queen_path="/content/queen_walk.mp4",
    max_frames=64,
    fps=16,
    decode_func=decode_latents_chunked,
    export_func=export_to_video,
    direction_amount=0.45
)

Loading /content/king_walk.mp4...


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Loading /content/queen_walk.mp4...


In [56]:
from IPython.display import Video, display

display(Video("output_king.mp4", embed=True, width=480))
display(Video("output_mid.mp4", embed=True, width=480))
display(Video("output_queen.mp4", embed=True, width=480))